# Matrix Norms and Condition Numbers - Examples

This notebook demonstrates matrix norms, condition numbers, and their applications.

## Contents
1. Vector Norms Review
2. Matrix Norms
3. Norm Properties
4. Condition Number
5. Ill-Conditioned Matrices
6. Regularization Effect
7. Spectral Normalization
8. Nuclear Norm
9. Norm Relationships
10. Condition Number and Optimization
11. Numerical Stability
12. Visualizations

In [ ]:
import numpy as np
from numpy.linalg import norm, cond, svd, inv, eig
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. Vector Norms Review

| Norm | Formula | Meaning |
|------|---------|--------|
| $L^1$ (Manhattan) | $\|\mathbf{x}\|_1 = \sum_i |x_i|$ | Sum of absolute values |
| $L^2$ (Euclidean) | $\|\mathbf{x}\|_2 = \sqrt{\sum_i x_i^2}$ | Straight-line distance |
| $L^\infty$ (Max) | $\|\mathbf{x}\|_\infty = \max_i |x_i|$ | Maximum component |
| $L^p$ | $\|\mathbf{x}\|_p = (\sum_i |x_i|^p)^{1/p}$ | General p-norm |

In [ ]:
x = np.array([3, -4, 0, 2])
print(f"x = {x}")

# L1 norm (Manhattan)
l1 = norm(x, 1)
print(f"\nL¹ norm: ||x||₁ = |3| + |-4| + |0| + |2| = {l1}")

# L2 norm (Euclidean)
l2 = norm(x, 2)
print(f"L² norm: ||x||₂ = √(9 + 16 + 0 + 4) = √29 = {l2:.4f}")

# L∞ norm (Max)
linf = norm(x, np.inf)
print(f"L∞ norm: ||x||_∞ = max(|3|, |-4|, |0|, |2|) = {linf}")

# General p-norm
for p in [3, 4]:
    lp = norm(x, p)
    print(f"L{p} norm: ||x||_{p} = {lp:.4f}")

In [ ]:
# Visualize unit balls for different norms
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

# Generate points on unit circles
theta = np.linspace(0, 2*np.pi, 1000)

# L1: diamond
ax = axes[0]
t = np.linspace(0, 2*np.pi, 5)
ax.plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], 'b-', linewidth=2)
ax.fill([1, 0, -1, 0], [0, 1, 0, -1], alpha=0.3)
ax.set_title('L¹ unit ball (diamond)', fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)

# L2: circle
ax = axes[1]
ax.plot(np.cos(theta), np.sin(theta), 'b-', linewidth=2)
ax.fill(np.cos(theta), np.sin(theta), alpha=0.3)
ax.set_title('L² unit ball (circle)', fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)

# L4: rounded square
ax = axes[2]
r4 = (np.abs(np.cos(theta))**4 + np.abs(np.sin(theta))**4)**(-0.25)
ax.plot(r4*np.cos(theta), r4*np.sin(theta), 'b-', linewidth=2)
ax.fill(r4*np.cos(theta), r4*np.sin(theta), alpha=0.3)
ax.set_title('L⁴ unit ball', fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)

# L∞: square
ax = axes[3]
ax.plot([1, 1, -1, -1, 1], [1, -1, -1, 1, 1], 'b-', linewidth=2)
ax.fill([1, 1, -1, -1], [1, -1, -1, 1], alpha=0.3)
ax.set_title('L∞ unit ball (square)', fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

## 2. Matrix Norms

### Induced (Operator) Norms

The induced norm measures maximum "stretching":

$$\|A\|_p = \max_{\mathbf{x} \neq 0} \frac{\|A\mathbf{x}\|_p}{\|\mathbf{x}\|_p}$$

### Common Matrix Norms

| Norm | Formula | Meaning |
|------|---------|--------|
| $\|A\|_1$ | Max column sum | Maximum L1-stretch |
| $\|A\|_\infty$ | Max row sum | Maximum L∞-stretch |
| $\|A\|_2$ (spectral) | $\sigma_{\max}$ | Maximum L2-stretch |
| $\|A\|_F$ (Frobenius) | $\sqrt{\sum a_{ij}^2}$ | Treats matrix as vector |
| $\|A\|_*$ (nuclear) | $\sum \sigma_i$ | Sum of singular values |

In [ ]:
A = np.array([[1, 2],
              [3, 4]])
print(f"A = \n{A}")

# L1 norm (max column sum)
l1_norm = norm(A, 1)
print(f"\n||A||₁ (max column sum):")
print(f"  Column 1: |1| + |3| = 4")
print(f"  Column 2: |2| + |4| = 6")
print(f"  ||A||₁ = max(4, 6) = {l1_norm}")

# L∞ norm (max row sum)
linf_norm = norm(A, np.inf)
print(f"\n||A||_∞ (max row sum):")
print(f"  Row 1: |1| + |2| = 3")
print(f"  Row 2: |3| + |4| = 7")
print(f"  ||A||_∞ = max(3, 7) = {linf_norm}")

In [ ]:
# SVD for spectral and Frobenius norms
U, S, Vt = svd(A)
print(f"Singular values: σ = {S}")

# Spectral norm (L2 induced)
l2_norm = norm(A, 2)
print(f"\n||A||₂ (spectral norm) = σ_max = {l2_norm:.4f}")

# Frobenius norm
fro_norm = norm(A, 'fro')
print(f"\n||A||_F (Frobenius norm):")
print(f"  Method 1: √(Σaᵢⱼ²) = √(1 + 4 + 9 + 16) = √30 = {np.sqrt(30):.4f}")
print(f"  Method 2: √(Σσᵢ²) = √({S[0]**2:.4f} + {S[1]**2:.4f}) = {np.sqrt(np.sum(S**2)):.4f}")
print(f"  NumPy: {fro_norm:.4f}")

# Nuclear norm
nuclear_norm = np.sum(S)
print(f"\n||A||_* (nuclear norm) = Σσᵢ = {S[0]:.4f} + {S[1]:.4f} = {nuclear_norm:.4f}")

## 3. Norm Properties

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[2, 0], [1, 1]])
c = 3

print("Norm Properties")
print("="*50)

# 1. Positive definite
print(f"\n1. Positive definite: ||A|| ≥ 0")
print(f"   ||A||_F = {norm(A, 'fro'):.4f} ≥ 0 ✓")
print(f"   ||0||_F = {norm(np.zeros((2,2)), 'fro')} = 0 ✓")

# 2. Homogeneity
print(f"\n2. Homogeneity: ||cA|| = |c|||A||")
print(f"   ||{c}A||_F = {norm(c * A, 'fro'):.4f}")
print(f"   |{c}| × ||A||_F = {abs(c) * norm(A, 'fro'):.4f} ✓")

# 3. Triangle inequality
print(f"\n3. Triangle inequality: ||A + B|| ≤ ||A|| + ||B||")
print(f"   ||A + B||_F = {norm(A + B, 'fro'):.4f}")
print(f"   ||A||_F + ||B||_F = {norm(A, 'fro') + norm(B, 'fro'):.4f} ✓")

# 4. Submultiplicativity
print(f"\n4. Submultiplicativity: ||AB|| ≤ ||A|| ||B||")
print(f"   ||AB||_F = {norm(A @ B, 'fro'):.4f}")
print(f"   ||A||_F × ||B||_F = {norm(A, 'fro') * norm(B, 'fro'):.4f} ✓")

## 4. Condition Number

The condition number measures sensitivity to perturbations:

$$\kappa(A) = \|A\| \|A^{-1}\| = \frac{\sigma_{\max}}{\sigma_{\min}}$$

**Interpretation:**
- $\kappa \approx 1$: Well-conditioned (stable)
- $\kappa \gg 1$: Ill-conditioned (sensitive to errors)
- $\kappa = \infty$: Singular (not invertible)

In [ ]:
# Well-conditioned (identity)
I = np.eye(2)
print("Well-conditioned (Identity):")
print(f"A = I, κ(A) = {cond(I):.4f}")

# Mildly ill-conditioned
B = np.array([[1, 0],
              [0, 0.1]])
U, S, Vt = svd(B)
print(f"\nMildly ill-conditioned:")
print(f"B = diag(1, 0.1)")
print(f"Singular values: {S}")
print(f"κ(B) = {S[0]}/{S[1]} = {cond(B):.1f}")

# Very ill-conditioned
C = np.array([[1, 1],
              [1, 1.0001]])
U, S, Vt = svd(C)
print(f"\nVery ill-conditioned:")
print(f"C = [[1, 1], [1, 1.0001]]")
print(f"Singular values: {S}")
print(f"κ(C) = {cond(C):.0f}")

In [ ]:
# Effect of ill-conditioning on solution
print("Effect on Linear System Solution")
print("="*50)

C = np.array([[1, 1],
              [1, 1.0001]])

b = np.array([2.0, 2.0])
x = np.linalg.solve(C, b)
print(f"\nSolving Cx = b where b = {b}")
print(f"Solution x = {x}")

# Perturb b slightly
b_perturbed = np.array([2.001, 2.0])
x_perturbed = np.linalg.solve(C, b_perturbed)

rel_change_b = norm(b_perturbed - b) / norm(b)
rel_change_x = norm(x_perturbed - x) / norm(x)

print(f"\nPerturbed b = {b_perturbed}")
print(f"Relative change in b: {rel_change_b*100:.3f}%")
print(f"\nNew solution = {x_perturbed}")
print(f"Relative change in x: {rel_change_x*100:.1f}%")
print(f"\nAmplification factor: {rel_change_x/rel_change_b:.0f}x ≈ κ(C)")

## 5. Ill-Conditioned Matrices

### Hilbert Matrix

The Hilbert matrix is notoriously ill-conditioned:

$$H_{ij} = \frac{1}{i + j - 1}$$

In [ ]:
def hilbert(n):
    """Generate n×n Hilbert matrix."""
    return np.array([[1/(i + j + 1) for j in range(n)] for i in range(n)])

print("Hilbert Matrix: H_ij = 1/(i + j - 1)")
print("="*50)

H3 = hilbert(3)
print(f"\nH₃ = \n{np.round(H3, 4)}")

print("\nCondition numbers for different sizes:")
for n in [3, 5, 7, 10, 12]:
    H = hilbert(n)
    kappa = cond(H)
    print(f"  n = {n:2d}: κ(H) = {kappa:.2e}")

In [ ]:
# Demonstrate error in solving Hx = b
print("\nSolving Hx = b for H₅:")
n = 5
H = hilbert(n)
x_true = np.ones(n)
b = H @ x_true

x_computed = np.linalg.solve(H, b)
error = norm(x_computed - x_true) / norm(x_true)

print(f"True solution: x = {x_true}")
print(f"Computed: x = {np.round(x_computed, 6)}")
print(f"Relative error: {error:.2e}")

print("\nFor H₁₀:")
n = 10
H = hilbert(n)
x_true = np.ones(n)
b = H @ x_true
x_computed = np.linalg.solve(H, b)
error = norm(x_computed - x_true) / norm(x_true)
print(f"Relative error: {error:.2e}")
print("Error grows exponentially with condition number!")

## 6. Regularization Effect on Condition Number

L2 regularization (ridge) adds $\lambda I$ to $X^TX$, improving conditioning.

In [ ]:
# Ill-conditioned data matrix
X = np.array([[1, 1, 1],
              [1, 1.001, 1],
              [1, 1, 1.001]])

XTX = X.T @ X
print(f"X^T X = \n{np.round(XTX, 4)}")
print(f"κ(X^T X) = {cond(XTX):.2e}")

print("\n--- Adding L2 regularization ---")
lambdas = [0.001, 0.01, 0.1, 1.0]
kappas = []

for lam in lambdas:
    XTX_reg = XTX + lam * np.eye(3)
    kappa = cond(XTX_reg)
    kappas.append(kappa)
    print(f"λ = {lam}: κ(X^TX + λI) = {kappa:.2e}")

print("\nRegularization dramatically improves conditioning!")

In [ ]:
# Visualize regularization effect
lambdas_plot = np.logspace(-4, 1, 50)
kappas_plot = [cond(XTX + lam * np.eye(3)) for lam in lambdas_plot]

plt.figure(figsize=(8, 5))
plt.loglog(lambdas_plot, kappas_plot, 'b-', linewidth=2)
plt.axhline(y=cond(XTX), color='r', linestyle='--', label=f'Original κ = {cond(XTX):.1e}')
plt.xlabel('Regularization λ', fontsize=11)
plt.ylabel('Condition Number κ', fontsize=11)
plt.title('Effect of L2 Regularization on Condition Number')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 7. Spectral Normalization

In GANs, spectral normalization constrains the discriminator's Lipschitz constant:

$$W_{\text{normalized}} = \frac{W}{\|W\|_2}$$

This ensures $\|W_{\text{normalized}}\|_2 = 1$.

In [ ]:
np.random.seed(42)

# Random weight matrix
W = np.random.randn(4, 4) * 2
print(f"Original weight matrix W:")
print(f"||W||₂ = {norm(W, 2):.4f}")

# Spectral normalization
W_normalized = W / norm(W, 2)
print(f"\nAfter spectral normalization (W / ||W||₂):")
print(f"||W_normalized||₂ = {norm(W_normalized, 2):.4f}")

In [ ]:
# Lipschitz constant analysis
print("Lipschitz Constant Analysis")
print("="*50)

# For linear layer: ||Wx - Wy|| / ||x - y|| ≤ ||W||_2
x = np.random.randn(4)
y = np.random.randn(4)

ratio_W = norm(W @ x - W @ y) / norm(x - y)
ratio_Wn = norm(W_normalized @ x - W_normalized @ y) / norm(x - y)

print(f"\nOriginal W:")
print(f"  ||Wx - Wy|| / ||x - y|| = {ratio_W:.4f}")
print(f"  Bounded by ||W||₂ = {norm(W, 2):.4f}")

print(f"\nNormalized W:")
print(f"  ||Wx - Wy|| / ||x - y|| = {ratio_Wn:.4f}")
print(f"  Bounded by ||W||₂ = 1.0")

print("\nSpectral normalization ensures 1-Lipschitz layers.")

## 8. Nuclear Norm

The nuclear norm is the sum of singular values:

$$\|A\|_* = \sum_i \sigma_i$$

Minimizing nuclear norm promotes low-rank solutions (matrix completion, recommender systems).

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 10]])  # Full rank

U, S, Vt = svd(A)

print(f"Matrix A:\n{A}")
print(f"\nSingular values: {np.round(S, 4)}")
print(f"\nNorms:")
print(f"  Nuclear ||A||_* = Σσᵢ = {np.sum(S):.4f}")
print(f"  Frobenius ||A||_F = √(Σσᵢ²) = {norm(A, 'fro'):.4f}")
print(f"  Spectral ||A||₂ = σ_max = {S[0]:.4f}")

In [ ]:
# Low-rank approximation
print("Low-Rank Approximation")
print("="*50)

for rank in [1, 2, 3]:
    A_approx = U[:, :rank] @ np.diag(S[:rank]) @ Vt[:rank, :]
    error = norm(A - A_approx, 'fro')
    nuclear = np.sum(S[:rank])
    print(f"\nRank-{rank} approximation:")
    print(f"  Nuclear norm: {nuclear:.4f}")
    print(f"  Error ||A - A_k||_F: {error:.4f}")
    if rank < 3:
        print(f"  (Error = √(Σ σᵢ² for i>{rank}))")

## 9. Norm Relationships

In [ ]:
A = np.array([[3, 0, 0],
              [0, 2, 0],
              [0, 0, 1]])

print(f"Diagonal matrix A = diag(3, 2, 1)\n{A}")

_, S, _ = svd(A)
norms = {
    '||A||₁': norm(A, 1),
    '||A||₂': norm(A, 2),
    '||A||_∞': norm(A, np.inf),
    '||A||_F': norm(A, 'fro'),
    '||A||_*': np.sum(S)
}

print("\nAll norms:")
for name, value in norms.items():
    print(f"  {name} = {value:.4f}")

print("\nRelationships:")
r = 3  # rank
print(f"||A||₂ ≤ ||A||_F: {norms['||A||₂']:.4f} ≤ {norms['||A||_F']:.4f} ✓")
print(f"||A||_F ≤ √r ||A||₂: {norms['||A||_F']:.4f} ≤ {np.sqrt(r) * norms['||A||₂']:.4f} ✓")
print(f"||A||₂ ≤ ||A||_* ≤ √r ||A||₂: {norms['||A||₂']:.4f} ≤ {norms['||A||_*']:.4f} ≤ {np.sqrt(r) * norms['||A||₂']:.4f} ✓")

## 10. Condition Number and Optimization

For quadratic $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^T H \mathbf{x}$, the condition number of the Hessian $H$ determines convergence speed.

In [ ]:
def gradient_descent(H, x0, lr, steps):
    """Gradient descent for quadratic f(x) = 0.5 x^T H x."""
    x = x0.copy()
    trajectory = [x.copy()]
    for _ in range(steps):
        grad = H @ x
        x = x - lr * grad
        trajectory.append(x.copy())
    return np.array(trajectory)

# Well-conditioned Hessian
H_good = np.array([[1, 0],
                   [0, 1]])

# Ill-conditioned Hessian
H_bad = np.array([[1, 0],
                  [0, 100]])

print(f"Well-conditioned H: κ = {cond(H_good):.0f}")
print(f"Ill-conditioned H: κ = {cond(H_bad):.0f}")

In [ ]:
x0 = np.array([10.0, 10.0])

# Well-conditioned: can use larger learning rate
traj_good = gradient_descent(H_good, x0, lr=0.5, steps=20)

# Ill-conditioned: need smaller learning rate for stability
traj_bad = gradient_descent(H_bad, x0, lr=0.01, steps=200)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot well-conditioned
ax = axes[0]
ax.plot(traj_good[:, 0], traj_good[:, 1], 'b.-', markersize=8)
ax.scatter([0], [0], c='red', s=100, zorder=5, label='Optimum')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title(f'Well-conditioned (κ = 1)\n20 steps, lr = 0.5')
ax.grid(True, alpha=0.3); ax.legend()
ax.set_xlim(-2, 12); ax.set_ylim(-2, 12)

# Plot ill-conditioned
ax = axes[1]
ax.plot(traj_bad[:, 0], traj_bad[:, 1], 'b.-', markersize=4, alpha=0.7)
ax.scatter([0], [0], c='red', s=100, zorder=5, label='Optimum')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title(f'Ill-conditioned (κ = 100)\n200 steps, lr = 0.01 (zigzag!)')
ax.grid(True, alpha=0.3); ax.legend()
ax.set_xlim(-2, 12); ax.set_ylim(-2, 12)

plt.tight_layout()
plt.show()

print(f"Final ||x|| for well-conditioned: {norm(traj_good[-1]):.2e}")
print(f"Final ||x|| for ill-conditioned: {norm(traj_bad[-1]):.2e}")

## 11. Numerical Stability

In [ ]:
np.random.seed(42)

# Well-conditioned system
A_good = np.array([[2, 1],
                   [1, 2]])
b = np.array([1.0, 1.0])

print("Numerical Stability Analysis")
print("="*50)

print(f"\nWell-conditioned A (κ = {cond(A_good):.2f}):")
x_true = np.linalg.solve(A_good, b)

# Add small perturbation
epsilon = 1e-10
A_noisy = A_good + epsilon * np.random.randn(2, 2)
x_noisy = np.linalg.solve(A_noisy, b)

rel_change_A = norm(A_noisy - A_good) / norm(A_good)
rel_change_x = norm(x_noisy - x_true) / norm(x_true)

print(f"Relative change in A: {rel_change_A:.2e}")
print(f"Relative change in x: {rel_change_x:.2e}")
print(f"Amplification: {rel_change_x / rel_change_A:.1f}x")

In [ ]:
# Ill-conditioned system
A_bad = np.array([[1, 1],
                  [1, 1.001]])

print(f"\nIll-conditioned A (κ = {cond(A_bad):.0f}):")
x_true_bad = np.linalg.solve(A_bad, b)

A_bad_noisy = A_bad + epsilon * np.random.randn(2, 2)
x_bad_noisy = np.linalg.solve(A_bad_noisy, b)

rel_change_A_bad = norm(A_bad_noisy - A_bad) / norm(A_bad)
rel_change_x_bad = norm(x_bad_noisy - x_true_bad) / norm(x_true_bad)

print(f"Relative change in A: {rel_change_A_bad:.2e}")
print(f"Relative change in x: {rel_change_x_bad:.2e}")
print(f"Amplification: {rel_change_x_bad / rel_change_A_bad:.0f}x")

print("\n" + "="*50)
print("Rule: Error can be amplified by factor κ(A)!")
print("If κ ≈ 10^k, expect to lose ~k digits of accuracy.")

## 12. Visualizations

In [ ]:
# Condition number geometry: how matrices stretch the unit circle
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

matrices = [
    ('κ = 1 (Identity)', np.eye(2)),
    ('κ = 3', np.array([[3, 0], [0, 1]])),
    ('κ = 10', np.array([[10, 0], [0, 1]]))
]

theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])

for ax, (name, A) in zip(axes, matrices):
    ellipse = A @ circle
    
    ax.plot(circle[0], circle[1], 'b--', alpha=0.3, label='Unit circle')
    ax.fill(ellipse[0], ellipse[1], alpha=0.3, color='red')
    ax.plot(ellipse[0], ellipse[1], 'r-', linewidth=2, label='A × circle')
    
    ax.set_xlim(-11, 11)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(name, fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('Condition Number: How Matrices Stretch the Unit Circle', fontsize=12)
plt.tight_layout()
plt.show()

print("Ill-conditioned matrices stretch circles into thin ellipses.")
print("Small perturbations along the thin direction → large changes!")

In [ ]:
# Singular value distribution for different matrices
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

np.random.seed(42)

# Random orthogonal
Q, _ = np.linalg.qr(np.random.randn(5, 5))
_, S1, _ = svd(Q)

# Random matrix
R = np.random.randn(5, 5)
_, S2, _ = svd(R)

# Hilbert matrix
H = hilbert(5)
_, S3, _ = svd(H)

data = [
    ('Orthogonal Q (κ=1)', S1),
    ('Random R', S2),
    ('Hilbert H₅', S3)
]

for ax, (name, S) in zip(axes, data):
    ax.bar(range(1, len(S)+1), S, color='steelblue')
    ax.set_xlabel('Singular Value Index')
    ax.set_ylabel('σᵢ')
    ax.set_title(f'{name}\nκ = {S[0]/S[-1]:.1e}')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

### Key Norms

| Norm | Formula | ML Application |
|------|---------|----------------|
| Spectral $\|A\|_2$ | $\sigma_{\max}$ | Lipschitz constant, spectral normalization |
| Frobenius $\|A\|_F$ | $\sqrt{\sum a_{ij}^2}$ | L2 regularization |
| Nuclear $\|A\|_*$ | $\sum \sigma_i$ | Matrix completion, low-rank |

### Condition Number

$$\kappa(A) = \frac{\sigma_{\max}}{\sigma_{\min}}$$

- $\kappa \approx 1$: Well-conditioned (stable)
- $\kappa \gg 1$: Ill-conditioned (lose ~$\log_{10}\kappa$ digits)

### ML Applications

```
Regularization:
├── L2 (Frobenius) → Weight decay
├── L1 → Sparsity
└── Nuclear → Low-rank

Stability:
├── Spectral normalization → Stabilizes GANs
├── Condition number → Affects optimization speed
└── Lipschitz bounds → Robustness
```